# Building a Google Drive Chatbot with Vertex AI RAG API

## Overview

This tutorial is based on Google own Retrieval-Augmented Generation (RAG) API doumentation you can find [here](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/rag-api).

This tutorial walks you through setting up a Retrieval-Augmented Generation (RAG) system using Google Cloud's Vertex AI. We will create a secure knowledge base (corpus), ingest document files directly from a Google Drive folder pathway, and build a chatbot capable of answering grounded questions with the latest Gemini models.

Vertex AI's **Retrieval-Augmented Generation (RAG) API** is a package that follows the RAG technique to a process that enhances the capabilities of large language models (LLMs) following the steps below.

First **ingest/intake data** from different data sources like local files, Cloud Storage, and Google Drive. Second **transform the data** by splitting it into chunks to allow better retrieval of related information to your questions, lower risk of the model having hallucinations, and decrease the amount of tokens that are used. Third **embedding the text** meaning the words or pieces of text will be represented by a number that states the relationship it has with other text, how similar they are. Fourth **indexing the data** by creating knowledge bases or **corpus** that will store the chunks and embeddings and be given ids to be queried and referenced from. Fifth now that our knowledge base set up we can now **retrieve relevant information** from it based on the question or query the user provides. Sixth the retrieved information, this being only the relevant chunks of data, is then given to the model for it to **generate a response** to the users question.






## Prerequisites

We assume you have access to **Vertex AI** and Google Drive and have enabled the APIs. If not go to the console then `APIs & Services > Enabled APIs & Services` search for **'Google Drive API'** then click 'Enable'. do the same for Vertex AI.

In this tutorial we will be using Google **gemini-2.0-flash** which doesn't need to be deployed but if you would like to use another model you choose one from the **Model Garden** using the console which will allow you to add a model to your model registry, create an endpoint (or use an existing one), and deploy the model all in one step. 

The last thing before we begin will to create a **Vertex AI RAG Data Service Agent** service account by going to `IAM` on the console then check mark **Include Google-provided role grant** if it not listed there then click grant access and add Vertex AI RAG Data Service Agent as a role.

## Learning objectives

For this tutorial we are creating a chatbot that will answer questions by gathering information from documents we have provided via Google Drive. The model we will be using today is a pretrained 'Gemini' model from GCP.

This tutorial will go over the following topics:
- Introducing RAG
- Creating a Vertex AI RAG Corpus
- Connecting our model to the RAG corpus


## Get started

### Step 1: Environment Initialization & Package Setup

First, install the unified Google Gen AI SDK alongside the necessary authentication utilities.

google-genai: This is the modern, unified Google Gen AI SDK. It replaces older, fragmented libraries (google-cloud-aiplatform, vertexai) with a single, clean API structure for interacting with both Gemini models and advanced Vertex platform services like RAG

In [ ]:
# Install the modern unified Google Gen AI SDK
!pip install --upgrade google-genai google-api-python-client google-auth-oauthlib google-auth-httplib2

- LOCATION = "europe-west3": Vertex AI RAG services utilize an automated Serverless Spanner back-end. This specific location (europe-west3) is designated to bypass Spanner region placement restrictions (allowlist constraints) that affect other zones, ensuring uninterrupted database allocation.
- vertexai.init() and aiplatform.init(): Both endpoints are initialized simultaneously using the same region parameters to guarantee that your high-level application calls match the lower-level netw

In [ ]:
# Import core dependencies and establish your workspace configuration constants
import os
import google.auth
import vertexai
from google.cloud import aiplatform

# Define target project parameters
PROJECT_ID = "your_project_id" 
LOCATION = "europe-west3"  # Regional routing context for Spanner endpoints
DRIVE_FOLDER_PATH = "your_google_drive_path"

# Initialize global Vertex AI context
vertexai.init(project=PROJECT_ID, location=LOCATION)
aiplatform.init(project=PROJECT_ID, location=LOCATION)

print("Environment correctly mapped to project: "+PROJECT_ID+" in "+LOCATION)

### Step 2: Establish User Credentials and API Sessions

Authenticate your interactive notebook session via Application Default Credentials (ADC) using wide Google client endpoint API scopes.

- google.auth.default(): This safely checks the internal system environment for active Google Cloud credentials (Application Default Credentials).
- scopes=[".../auth/cloud-platform"]: Instead of requesting narrow or restrictive access permissions, this scope acts as an umbrella. It allows the Python environment to securely manage complex cross-service handshakes—such as reading data from Google Drive and writing it directly to a Vertex AI data vector corpus.

In [ ]:
# Setup authentication scopes for the notebook session
credentials, project_id = google.auth.default(
    scopes=["https://googleapis.com"]
)

print(" Successfully authenticated via Application Default Credentials (ADC).")


### Step 3: Create the Serverless RAG Corpus

We initialize a dedicated text knowledge base container (RagCorpus) using Google's foundational text embedding model configuration.
python

- text-embedding-004: This model represents Google’s premier text vectorization engine. It converts raw alphanumeric text fragments into complex numerical math arrays (embeddings). It is selected for its high dimensional accuracy and semantic understanding across medical and technical literature.
- rag.create_corpus(): This command creates a serverless vector database index behind the scenes. It handles all scaling, memory orchestration, and data structures automatically without requiring manual setup.
- corpus.name: The database outputs a unique global resource path string (e.g., projects/.../locations/.../ragCorpora/12345). We capture this text value to direct all future data uploads and retrieval lookups to the correct index.


In [ ]:
from vertexai.preview import rag

# 1. Define standard Text Embedding model properties
embedding_config = rag.EmbeddingModelConfig(
    publisher_model="publishers/google/models/text-embedding-004"
)

# 2. Build the Serverless RagCorpus structure dynamically
print("Creating a fresh Serverless RagCorpus")
corpus = rag.create_corpus(
    display_name="tutorial_dynamic_corpus",
    embedding_model_config=embedding_config
)
    
# Extract and save the system-generated resource name
CORPUS_RESOURCE_NAME = corpus.name
print(" Corpus created successfully!")
print(" Resource Name: "+CORPUS_RESOURCE_NAME)

### Optional - Download articles

For this tutorial we will be downloading scientific articles from the NIH RECOVER program, which our model will then use as references to answer our questions about COVID. If you have your own file on your drive feel free to use those.

In [ ]:
#urls list of articles
articles_urls = ['https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10781091/pdf/pone.0285645.pdf', 
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10219649/pdf/elife-86014.pdf',
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10734909/pdf/pone.0285351.pdf',
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10684592/pdf/41598_2023_Article_47655.pdf', 
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10601201/pdf/12889_2023_Article_16916.pdf',
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10516599/pdf/elife-86043.pdf',
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10620090/pdf/41586_2023_Article_6651.pdf',
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10414557/pdf/pone.0289774.pdf',
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10355333/pdf/aids-37-1565.pdf',
                 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10289397/pdf/pone.0286297.pdf']


Now well use a for loop to run `subprocess.run` to download each article.

In [ ]:
import subprocess
for url in articles_urls:
    subprocess.run(f'wget --user-agent="Chrome" {url}', shell=True, executable="/bin/bash")

Create a folder within your drive and upload these docs into your new drive folder. If you are using Jupyter Lab you can download the docs by right clicking them in the File Browser and selecting download. 

Then you will need to go to **IAM** on the console, check mark where it says **Include Google-provided role grants**, and copy the principcal email address with the **Vertex AI RAG Data Service Agent** role.

![gdrive1](../../images/gdrive1.png)

Then go to your Google Drive and share the folder you just created with that email address and add **Viewer permissions**.
![gdrive2](../../images/gdrive2.png)
![gdrive1.5](../../images/gdrive1.5.png)
![gdrive3](../../images/gdrive3.png)







### Step 4: Ingest Document Files from Google Drive

Trigger an asynchronous ingestion pipeline to read, parse, and split files inside your target Google Drive folder pathway directly into your vector knowledge database.

- rag.import_files(): This acts as an automated pipeline worker. It opens the target Google Drive directory, reads through complex files (like PDFs or Word docs), segments long paragraphs into small, manageable sentences (chunking), and runs them through the embedding model before saving them to the database index.
- paths=[DRIVE_FOLDER_PATH]: The path parameter expects an absolute asset list format. Passing the single URL root folder instructs Vertex AI to process all document assets found within that directory at once.

In [ ]:
print("Initiating Google Drive file ingestion job")
    
# Trigger parsing framework via explicit regional routing paths
response = rag.import_files(corpus_name=CORPUS_RESOURCE_NAME,  paths=[DRIVE_FOLDER_PATH] )
    
print(" Successfully triggered import job")
print(" Imported files count: "+{response.imported_rag_files_count)

### Step 5: Verify Active Ingested Knowledge Files

List the document corpus contents to confirm processing statuses and pull deep system file metadata details safely.

- rag.list_files(): Since document data processing runs asynchronously in the cloud background, this check confirms that your documents are fully processed.
- file.file_status.state.name: Returns structural execution states (like ACTIVE, PROCESSING, or FAILED). You want to ensure files show as ACTIVE before running automated chatbot queries, verifying that text chunks have been successfully indexed.


In [ ]:
# Query files allocated to our target corpus registry
files = rag.list_files(corpus_name=CORPUS_RESOURCE_NAME)
    
print(" Registered Corpus Documents:\n" + "-"*40)
for file in files:
    print(" File Name: "+file.display_name)
    print(" Resource ID: "+file.name)
    print(" Processing State: "+file.file_status.state.name)


### Step 6: Grounded Generation with Gemini

Connect the corpus directory as a retrieval engine asset using unified genai.Client schema tools. The model will automatically inject context chunks to answer inquiries reliably without hallucinating.

- vertexai=True: This parameter tells the unified genai.Client to use backend cloud enterprise systems (like Vertex AI indexes and billing boundaries) rather than the standard public consumer developer API.
- types.VertexRagStore(): This configuration safely links your vector database to the Gemini text inference pipeline as a native tool attachment.
- top_k=4: Why 4? The top_k parameter controls the maximum number of text context chunks retrieved from the database to answer a question. Setting this to 4 balances providing rich, comprehensive context (e.g., matching across multiple documents) with optimizing execution costs by avoiding unnecessary token usage.
- model="gemini-3.5-flash": This production-tier multimodal model is selected for its high speed, low latency, large context window capability, and strong performance when reasoning over complex text contexts.


In [ ]:
from google import genai
from google.genai import types

# 1. Initialize the modern GenAI Client mapping to your deployment region
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

# 2. Configure the RAG engine utility matching the corpus mapping schema
rag_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[{"rag_corpus": CORPUS_RESOURCE_NAME}],
            rag_retrieval_config=types.RagRetrievalConfig(top_k=4)
        )
    )
)

user_query = "What is Long COVID according to the provided documentation?"

print(" Dispatching query to gemini-3.5-flash")
    
# 3. Call the model and pass our retrieval tool config
response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=user_query,
        config=types.GenerateContentConfig(tools=[rag_tool])
    )
    
print("\ Grounded Response from Gemini:")
print(response.text)


### Clean Up

It is highly recommended to perform a cleanup after completing this tutorial. Because Vertex AI's RAG API provisions serverless resources (like storage allocations and indexing engines) behind the scenes, leaving them active can result in ongoing cloud costs.

To clean up, execute the following code block in your notebook to delete the RAG corpus and all its ingested files:

In [ ]:
from vertexai.preview import rag

print("Commencing cleanup: Deleting RAG Corpus "+CORPUS_RESOURCE_NAME)
    
# Force-deleting the corpus automatically removes all nested chunks and documents
rag.delete_corpus(CORPUS_RESOURCE_NAME)
    
print("Cleanup complete! Corpus and all associated data vectors have been successfully removed.")


- Cost Management: Google Cloud charges for vector data storage and backend indexing resources. Deleting the corpus stops the billing meter for this environment.
- Workspace Hygiene: It prevents your GCP project from becoming cluttered with temporary testing corpora (projects/.../ragCorpora/...), ensuring clean asset boundaries when you transition to production systems.
